In [32]:
import time
import gc
import argparse
import sys

def benchmark_numpy(n, runs):
    """Runs the matrix multiplication benchmark using standard NumPy on the CPU."""
    import numpy as np
    
    print(f"--- NumPy (CPU) Benchmark ---")
    print(f"Multiplying two {n}×{n} matrices ({runs} runs)\n")

    # 1. Generate data ONCE before the timing loop.
    print(f"Generating two {n}x{n} random matrices on CPU...")
    A = np.random.rand(n, n).astype(np.float32)
    B = np.random.rand(n, n).astype(np.float32)

    # 2. Perform one untimed warm-up run.
    print("Performing warm-up run...")
    _ = np.matmul(A, B)
    print("Warm-up complete.\n")

    # 3. Perform the timed runs.
    times = []
    for i in range(runs):
        start = time.time()
        # The operation being timed. The @ operator is a convenient
        # shorthand for np.matmul.
        C = A @ B
        end = time.time()

        duration = end - start
        times.append(duration)
        print(f"Run {i+1}: time = {duration:.4f}s")
        del C # Clean up the result matrix
        gc.collect()

    avg = sum(times) / len(times)
    print(f"\nNumPy average: {avg:.4f}s\n")
    return avg

def benchmark_cupy(n, runs):
    """Runs the matrix multiplication benchmark using standard NumPy on the CPU."""
    import cupy as cp
    
    print(f"--- CuPy (GPU) Benchmark ---")
    print(f"Multiplying two {n}×{n} matrices ({runs} runs)\n")

    # 1. Generate data ONCE before the timing loop.
    print(f"Generating two {n}x{n} random matrices on GPU...")
    A = cp.random.rand(n, n).astype(cp.float32)
    B = cp.random.rand(n, n).astype(cp.float32)

    # 2. Perform one untimed warm-up run.
    print("Performing warm-up run...")
    _ = cp.matmul(A, B)
    print("Warm-up complete.\n")

    # 3. Perform the timed runs.
    times = []
    for i in range(runs):
        start = time.time()
        # The operation being timed. The @ operator is a convenient
        # shorthand for np.matmul.
        C = A @ B
        end = time.time()

        duration = end - start
        times.append(duration)
        print(f"Run {i+1}: time = {duration:.4f}s")
        del C # Clean up the result matrix
        gc.collect()

    avg = sum(times) / len(times)
    print(f"\nCuPy average: {avg:.4f}s\n")
    return avg
    
def benchmark_cunumeric(n, runs):
    """Runs the matrix multiplication benchmark using cuNumeric on the GPU."""
    import cupynumeric as cn
    import numpy as np # Import numpy for the canonical sync
    
    print(f"--- cuNumeric (GPU) Benchmark ---")
    print(f"Multiplying two {n}×{n} matrices ({runs} runs)\n")

    # 1. Generate data ONCE on the GPU before the timing loop.
    print(f"Generating two {n}x{n} random matrices on GPU...")
    A = cn.random.rand(n, n).astype(np.float32)
    B = cn.random.rand(n, n).astype(np.float32)

    # 2. Perform a crucial untimed warm-up run for JIT compilation.
    print("Performing warm-up run...")
    C_warmup = cn.matmul(A, B)
    # The best practice for synchronization: force a copy back to the CPU.
    _ = np.array(C_warmup)
    print("Warm-up complete.\n")

    # 3. Perform the timed runs.
    times = []
    for i in range(runs):
        start = time.time()
        
        # Launch the operation on the GPU
        C = A @ B
        
        # Synchronize by converting the result to a host-side NumPy array.
        np.array(C)

        end = time.time()

        duration = end - start
        times.append(duration)
        print(f"Run {i+1}: time = {duration:.4f}s")
        del C
        gc.collect()

    avg = sum(times) / len(times)
    print(f"\ncuNumeric average: {avg:.4f}s\n")
    return avg

In [33]:
benchmark_numpy(3000, 5)
benchmark_cupy(3000, 5)
benchmark_cunumeric(3000, 5)

--- NumPy (CPU) Benchmark ---
Multiplying two 3000×3000 matrices (5 runs)

Generating two 3000x3000 random matrices on CPU...
Performing warm-up run...
Warm-up complete.

Run 1: time = 0.0945s
Run 2: time = 0.0943s
Run 3: time = 0.0947s
Run 4: time = 0.0950s
Run 5: time = 0.0956s

NumPy average: 0.0948s

--- CuPy (GPU) Benchmark ---
Multiplying two 3000×3000 matrices (5 runs)

Generating two 3000x3000 random matrices on GPU...
Performing warm-up run...
Warm-up complete.

Run 1: time = 0.0000s
Run 2: time = 0.0001s
Run 3: time = 0.0001s
Run 4: time = 0.0001s
Run 5: time = 0.0001s

CuPy average: 0.0001s

--- cuNumeric (GPU) Benchmark ---
Multiplying two 3000×3000 matrices (5 runs)

Generating two 3000x3000 random matrices on GPU...
Performing warm-up run...
Warm-up complete.

Run 1: time = 0.0155s
Run 2: time = 0.0135s
Run 3: time = 0.0118s
Run 4: time = 0.0121s
Run 5: time = 0.0116s

cuNumeric average: 0.0129s



0.012902450561523438

In [34]:
import time
import gc
import argparse
import sys

# --- Reusable Training Function ---
# By putting the training loop in its own function, we avoid code duplication.
# The `np` argument allows us to pass in either the numpy or cupynumeric module.
def train_logistic_regression(np, X, y, iters, alpha):
    """Performs a set number of gradient descent iterations."""
    # Ensure w starts on the correct device (CPU or GPU)
    w = np.zeros(X.shape[1])
    
    for _ in range(iters):
        z = X.dot(w)
        p = 1.0 / (1.0 + np.exp(-z))
        grad = X.T.dot(p - y) / X.shape[0]
        w -= alpha * grad
    
    return w

def benchmark_numpy(n_samples, n_features, iters, alpha):
    """Runs the logistic regression benchmark using standard NumPy on the CPU."""
    import numpy as np
    
    print(f"--- NumPy (CPU) Benchmark ---")
    print(f"Training on {n_samples} samples, {n_features} features for {iters} iterations\n")

    # 1. Generate data ONCE before the timing loop.
    print("Generating random dataset on CPU...")
    X = np.random.rand(n_samples, n_features)
    y = (np.random.rand(n_samples) > 0.5).astype(np.float64)

    # 2. Perform one untimed warm-up run.
    print("Performing warm-up run...")
    _ = train_logistic_regression(np, X, y, iters, alpha)
    print("Warm-up complete.\n")

    # 3. Perform the timed runs.
    times = []
    for i in range(5):
        start = time.time()
        # The operation being timed
        _ = train_logistic_regression(np, X, y, iters, alpha)
        end = time.time()

        duration = end - start
        times.append(duration)
        print(f"Run {i+1}: time = {duration:.3f}s")
        gc.collect()

    avg = sum(times) / len(times)
    print(f"\nNumPy average: {avg:.3f}s\n")
    return avg

def benchmark_cupy(n_samples, n_features, iters, alpha):
    """Runs the logistic regression benchmark using CuPy on the GPU."""
    import cupy as cp # Also import numpy for the canonical synchronization
    
    print(f"--- CuPy (GPU) Benchmark ---")
    print(f"Training on {n_samples} samples, {n_features} features for {iters} iterations\n")

    # 1. Generate data ONCE on the GPU before the timing loop.
    print("Generating random dataset on GPU...")
    X = cp.random.rand(n_samples, n_features)
    y = (cp.random.rand(n_samples) > 0.5).astype(cp.float64)

    # 2. Perform a crucial untimed warm-up run for JIT compilation.
    print("Performing warm-up run...")
    w_warmup = train_logistic_regression(cp, X, y, iters, alpha)
    # The best practice for synchronization: force a copy back to the CPU.
    _ = cp.array(w_warmup)
    print("Warm-up complete.\n")

    # 3. Perform the timed runs.
    times = []
    for i in range(5):
        start = time.time()
        
        # Launch the operation on the GPU
        w = train_logistic_regression(cp, X, y, iters, alpha)
        
        # Synchronize by converting the final result back to a NumPy array.
        cp.asnumpy(w)

        end = time.time()

        duration = end - start
        times.append(duration)
        print(f"Run {i+1}: time = {duration:.3f}s")
        del w
        gc.collect()

    avg = sum(times) / len(times)
    print(f"\nCuPy average: {avg:.3f}s\n")
    return avg

def benchmark_cunumeric(n_samples, n_features, iters, alpha):
    """Runs the logistic regression benchmark using cuNumeric on the GPU."""
    import cupynumeric as cn
    import numpy as np # Also import numpy for the canonical synchronization
    
    print(f"--- cuNumeric (GPU) Benchmark ---")
    print(f"Training on {n_samples} samples, {n_features} features for {iters} iterations\n")

    # 1. Generate data ONCE on the GPU before the timing loop.
    print("Generating random dataset on GPU...")
    X = cn.random.rand(n_samples, n_features)
    y = (cn.random.rand(n_samples) > 0.5).astype(np.float64)

    # 2. Perform a crucial untimed warm-up run for JIT compilation.
    print("Performing warm-up run...")
    w_warmup = train_logistic_regression(cn, X, y, iters, alpha)
    # The best practice for synchronization: force a copy back to the CPU.
    _ = np.array(w_warmup)
    print("Warm-up complete.\n")

    # 3. Perform the timed runs.
    times = []
    for i in range(5):
        start = time.time()
        
        # Launch the operation on the GPU
        w = train_logistic_regression(cn, X, y, iters, alpha)
        
        # Synchronize by converting the final result back to a NumPy array.
        np.array(w)

        end = time.time()

        duration = end - start
        times.append(duration)
        print(f"Run {i+1}: time = {duration:.3f}s")
        del w
        gc.collect()

    avg = sum(times) / len(times)
    print(f"\ncuNumeric average: {avg:.3f}s\n")
    return avg


In [35]:
benchmark_numpy(2_000_000, 10, 500, 0.1)
benchmark_cupy(2_000_000, 10, 500, 0.1)
benchmark_cunumeric(2_000_000, 10, 500, 0.1)

--- NumPy (CPU) Benchmark ---
Training on 2000000 samples, 10 features for 500 iterations

Generating random dataset on CPU...
Performing warm-up run...
Warm-up complete.

Run 1: time = 10.857s
Run 2: time = 10.932s
Run 3: time = 11.173s
Run 4: time = 11.290s
Run 5: time = 10.900s

NumPy average: 11.031s

--- CuPy (GPU) Benchmark ---
Training on 2000000 samples, 10 features for 500 iterations

Generating random dataset on GPU...
Performing warm-up run...
Warm-up complete.

Run 1: time = 1.723s
Run 2: time = 1.459s
Run 3: time = 1.460s
Run 4: time = 1.459s
Run 5: time = 1.460s

CuPy average: 1.512s

--- cuNumeric (GPU) Benchmark ---
Training on 2000000 samples, 10 features for 500 iterations

Generating random dataset on GPU...
Performing warm-up run...
Warm-up complete.

Run 1: time = 1.565s
Run 2: time = 1.560s
Run 3: time = 1.546s
Run 4: time = 1.539s
Run 5: time = 1.544s

cuNumeric average: 1.551s



1.5511117458343506

In [40]:
import time
import gc
import argparse
import sys # Import sys to check arguments

# Note: The library imports (numpy and cupynumeric) are now done *inside*
# their respective functions to keep them separate and avoid import errors.

def benchmark_numpy(n, runs):
    """Runs the linear solve benchmark using standard NumPy on the CPU."""
    import numpy as np

    print(f"--- NumPy (CPU) Benchmark ---")
    print(f"Solving {n}×{n} A x = b ({runs} runs)\n")

    # 1. Generate data ONCE before the timing loop.
    print("Generating random system on CPU...")
    A = np.random.randn(n, n).astype(np.float32)
    b = np.random.randn(n).astype(np.float32)

    # 2. Perform one untimed warm-up run. This is good practice even for
    # the CPU to ensure caches are warm and any one-time setup is done.
    print("Performing warm-up run...")
    _ = np.linalg.solve(A, b)
    print("Warm-up complete.\n")

    # 3. Perform the timed runs.
    times = []
    for i in range(runs):
        start = time.time()
        # The operation being timed
        x = np.linalg.solve(A, b)
        end = time.time()

        duration = end - start
        times.append(duration)
        print(f"Run {i+1}: time = {duration:.6f}s")
        # Clean up the result to be safe with memory
        del x
        gc.collect()

    avg = sum(times) / len(times)
    print(f"\nNumPy average: {avg:.6f}s\n")
    return avg

def benchmark_cupy(n, runs):
    """Runs the linear solve benchmark using standard NumPy on the CPU."""
    import cupy as cp

    print(f"--- CuPy (GPU) Benchmark ---")
    print(f"Solving {n}×{n} A x = b ({runs} runs)\n")

    # 1. Generate data ONCE before the timing loop.
    print("Generating random system on GPU...")
    A = cp.random.randn(n, n).astype(cp.float32)
    b = cp.random.randn(n).astype(cp.float32)

    # 2. Perform one untimed warm-up run. This is good practice even for
    # the CPU to ensure caches are warm and any one-time setup is done.
    print("Performing warm-up run...")
    _ = cp.linalg.solve(A, b)
    print("Warm-up complete.\n")

    # 3. Perform the timed runs.
    times = []
    for i in range(runs):
        start = time.time()
        # The operation being timed
        x = cp.linalg.solve(A, b)
        end = time.time()

        duration = end - start
        times.append(duration)
        print(f"Run {i+1}: time = {duration:.6f}s")
        # Clean up the result to be safe with memory
        del x
        gc.collect()

    avg = sum(times) / len(times)
    print(f"\nCuPy average: {avg:.6f}s\n")
    return avg

def benchmark_cunumeric(n, runs):
    """Runs the linear solve benchmark using cuNumeric on the GPU."""
    import cupynumeric as cn
    import numpy as np # Also import numpy for the canonical synchronization

    print(f"--- cuNumeric (GPU) Benchmark ---")
    print(f"Solving {n}×{n} A x = b ({runs} runs)\n")

    # 1. Generate data ONCE on the GPU before the timing loop.
    # This ensures we are not timing the data transfer in our main loop.
    print("Generating random system on GPU...")
    A = cn.random.randn(n, n).astype(np.float32)
    b = cn.random.randn(n).astype(np.float32)

    # 2. Perform a crucial untimed warm-up run. This handles JIT
    # compilation and other one-time GPU setup costs.
    print("Performing warm-up run...")
    x_warmup = cn.linalg.solve(A, b)
    # The best practice for synchronization: force a copy back to the CPU.
    _ = np.array(x_warmup)
    print("Warm-up complete.\n")

    # 3. Perform the timed runs.
    times = []
    for i in range(runs):
        start = time.time()

        # Launch the operation on the GPU
        x = cn.linalg.solve(A, b)

        # Synchronize by converting the result to a host-side NumPy array.
        # This is guaranteed to block until the GPU has finished.
        np.array(x)

        end = time.time()

        duration = end - start
        times.append(duration)
        print(f"Run {i+1}: time = {duration:.6f}s")
        # Clean up the GPU array result
        del x
        gc.collect()

    avg = sum(times) / len(times)
    print(f"\ncuNumeric average: {avg:.6f}s\n")
    return avg


In [41]:
benchmark_numpy(3000, 5)
benchmark_cupy(3000, 5)
benchmark_cunumeric(3000, 5)

--- NumPy (CPU) Benchmark ---
Solving 3000×3000 A x = b (5 runs)

Generating random system on CPU...
Performing warm-up run...
Warm-up complete.

Run 1: time = 0.107796s
Run 2: time = 0.103350s
Run 3: time = 0.103581s
Run 4: time = 0.102433s
Run 5: time = 0.103555s

NumPy average: 0.104143s

--- CuPy (GPU) Benchmark ---
Solving 3000×3000 A x = b (5 runs)

Generating random system on GPU...
Performing warm-up run...
Warm-up complete.

Run 1: time = 0.000434s
Run 2: time = 0.000649s
Run 3: time = 0.000485s
Run 4: time = 0.000492s
Run 5: time = 0.000484s

CuPy average: 0.000509s

--- cuNumeric (GPU) Benchmark ---
Solving 3000×3000 A x = b (5 runs)

Generating random system on GPU...
Performing warm-up run...
Warm-up complete.

Run 1: time = 0.012131s
Run 2: time = 0.013216s
Run 3: time = 0.012591s
Run 4: time = 0.012284s
Run 5: time = 0.012117s

cuNumeric average: 0.012468s



0.012467670440673827

In [ ]:
# benchmark_sort.py
import time
import sys
import gc

# Array size
n = 30_000_000 # 30 million elements

def benchmark_numpy():
    import numpy as np
    print(f"Sorting an array of {n} elements with NumPy (5 runs)\n")

    times = []
    for i in range(5):
        data = np.random.randn(n).astype(np.float32)
        start = time.time()
        _ = np.sort(data)
        end = time.time()

        duration = end - start
        times.append(duration)
        print(f"Run {i+1}: time = {duration:.6f}s")
        del data
        gc.collect()

    avg = sum(times) / len(times)
    print(f"\nNumPy average: {avg:.6f}s\n")

def benchmark_cupy():
    import cupy as cp
    print(f"Sorting an array of {n} elements with CuPy (5 runs)\n")

    times = []
    for i in range(5):
        data = cp.random.randn(n).astype(cp.float32)
        start = time.time()
        _ = cp.sort(data)
        end = time.time()

        duration = end - start
        times.append(duration)
        print(f"Run {i+1}: time = {duration:.6f}s")
        del data
        gc.collect()

    avg = sum(times) / len(times)
    print(f"\nCuPy average: {avg:.6f}s\n")

def benchmark_cunumeric():
    import cupynumeric as np
    print(f"Sorting an array of {n} elements with cuNumeric (5 runs)\n")

    times = []
    for i in range(5):
        data = np.random.randn(n).astype(np.float32)
        start = time.time()
        _ = np.sort(data)
        # Force GPU sync
        _ = np.linalg.norm(np.zeros(()))
        end = time.time()

        duration = end - start
        times.append(duration)
        print(f"Run {i+1}: time = {duration:.6f}s")
        del data
        gc.collect()
        _ = np.linalg.norm(np.zeros(()))

    avg = sum(times) / len(times)
    print(f"\ncuNumeric average: {avg:.6f}s\n")


In [30]:
benchmark_numpy()
benchmark_cupy()
benchmark_cunumeric()

Sorting an array of 30000000 elements with NumPy (5 runs)

Run 1: time = 1.692365s
Run 2: time = 1.732840s
Run 3: time = 1.789760s
Run 4: time = 1.779573s
Run 5: time = 1.803592s

NumPy average: 1.759626s

Sorting an array of 30000000 elements with CuPy (5 runs)

Run 1: time = 0.019050s
Run 2: time = 0.009980s
Run 3: time = 0.008960s
Run 4: time = 0.008973s
Run 5: time = 0.009012s

CuPy average: 0.011195s

Sorting an array of 30000000 elements with cuNumeric (5 runs)

Run 1: time = 0.000292s
Run 2: time = 0.000172s
Run 3: time = 0.000186s
Run 4: time = 0.000172s
Run 5: time = 0.000196s

cuNumeric average: 0.000204s

